<a href="https://colab.research.google.com/github/Ming-flowbetter/Machine-Learning/blob/main/lec07_nn_backprop_optimization_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 07 demo: backprop, vanishing gradients, and optimizers from scratch

CSCI 567 --- Machine Learning

Three self-contained parts, all NumPy-only (no scikit-learn/PyTorch):

1. **Gradient check**: a small feedforward network's manual forward pass and backpropagation
   (Lecture 07), verified against numerical differentiation -- the standard way to
   confirm a hand-derived backprop implementation is actually correct.
2. **Vanishing gradients**: a deep network's backward error signal $\delta^{(l)}$,
   measured layer by layer, comparing a saturating activation (sigmoid) against a non-saturating
   one (ReLU).
3. **GD vs. momentum vs. Adam**: all three optimizers on the Rosenbrock function, a
   classic ill-conditioned, non-convex test surface with a narrow curved valley.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

## Part 1: Gradient check

### 1.1 A small two-layer network

Input dimension $d=4$, hidden width $K=6$, scalar output, sigmoid activation, squared loss
averaged over a batch of $n=5$ synthetic examples.
This is exactly the lecture notes' general-network recursion with $L=1$ hidden layer:
$$\mathbf{z}^{(1)} = (W^{(1)})^\top\mathbf{x} + b^{(1)}, \qquad \mathbf{x}^{(1)} = \sigma(\mathbf{z}^{(1)}), \qquad
\hat y = (W^{(2)})^\top\mathbf{x}^{(1)} + b^{(2)}.$$

In [ ]:
rng = np.random.default_rng(0)

d, K, out = 4, 6, 1
n = 5

X = rng.normal(size=(n, d))
Y = rng.normal(size=(n, out))

W1 = rng.normal(scale=0.5, size=(d, K))
b1 = rng.normal(scale=0.5, size=(K,))
W2 = rng.normal(scale=0.5, size=(K, out))
b2 = rng.normal(scale=0.5, size=(out,))

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1 - s)

def forward(X, W1, b1, W2, b2):
    """Forward pass: compute and return every cached activation ."""
    Z1 = X @ W1 + b1          # (n, K)
    X1 = sigmoid(Z1)          # (n, K)
    Z2 = X1 @ W2 + b2         # (n, out) -- linear output layer
    Yhat = Z2
    return Z1, X1, Yhat

def loss_fn(X, Y, W1, b1, W2, b2):
    _, _, Yhat = forward(X, W1, b1, W2, b2)
    return 0.5 * np.mean(np.sum((Y - Yhat) ** 2, axis=1))

print("initial loss:", loss_fn(X, Y, W1, b1, W2, b2))

### 1.2 Backpropagation, exactly the lecture's recursion

$\delta^{(2)} = \hat y - y$ at the (linear) output layer, then
$\delta^{(1)} = (W^{(2)}\delta^{(2)})\odot\sigma'(\mathbf{z}^{(1)})$, and every parameter gradient
follows directly: $\partial\mathcal{L}/\partial W^{(l)}_{ij} = \delta_j^{(l)}x_i^{(l-1)}$,
$\partial\mathcal{L}/\partial b_j^{(l)} = \delta_j^{(l)}$.

In [ ]:
def backprop(X, Y, W1, b1, W2, b2):
    n = X.shape[0]
    Z1, X1, Yhat = forward(X, W1, b1, W2, b2)

    Delta2 = (Yhat - Y) / n                        # output-layer error signal, averaged over the batch
    gW2 = X1.T @ Delta2                            # dL/dW2
    gb2 = Delta2.sum(axis=0)                       # dL/db2

    Delta1 = (Delta2 @ W2.T) * sigmoid_prime(Z1)   # backward recursion: delta^(1) = (W2 delta^(2)) * g'(z^(1))
    gW1 = X.T @ Delta1                             # dL/dW1
    gb1 = Delta1.sum(axis=0)                       # dL/db1

    return gW1, gb1, gW2, gb2

gW1, gb1, gW2, gb2 = backprop(X, Y, W1, b1, W2, b2)
print("analytic gradient shapes:", gW1.shape, gb1.shape, gW2.shape, gb2.shape)

### 1.3 The gradient check

Perturb every parameter entry by $\pm\varepsilon$ and use the central-difference estimate
$(\mathcal{L}(\theta+\varepsilon)-\mathcal{L}(\theta-\varepsilon))/(2\varepsilon)$ -- if backprop is
implemented correctly, this must match the analytic gradient to close to machine precision.

In [ ]:
def numerical_grad(param, eps=1e-5):
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    for _ in it:
        idx = it.multi_index
        orig = param[idx]
        param[idx] = orig + eps
        lp = loss_fn(X, Y, W1, b1, W2, b2)
        param[idx] = orig - eps
        lm = loss_fn(X, Y, W1, b1, W2, b2)
        param[idx] = orig
        grad[idx] = (lp - lm) / (2 * eps)
    return grad

numW1 = numerical_grad(W1)
numb1 = numerical_grad(b1)
numW2 = numerical_grad(W2)
numb2 = numerical_grad(b2)

print("=== gradient check: analytic (backprop) vs. numerical ===")
worst = 0.0
for name, ana, num in [("W1", gW1, numW1), ("b1", gb1, numb1), ("W2", gW2, numW2), ("b2", gb2, numb2)]:
    diff = np.max(np.abs(ana - num))
    rel = diff / (np.max(np.abs(num)) + 1e-12)
    worst = max(worst, rel)
    print(f"{name}: max abs diff = {diff:.3e}, max rel diff = {rel:.3e}")
print(f"\nworst-case relative difference across all parameters: {worst:.3e}")
print("PASS -- matches to numerical precision" if worst < 1e-6 else "FAIL -- check the derivation")

**Result:** every analytic gradient matches its numerical counterpart to about $10^{-10}$ relative
error -- essentially machine precision for a central-difference check with $\varepsilon=10^{-5}$.
Backpropagation really does compute the exact gradient the chain-rule derivation claims it does.

## Part 2: Vanishing gradients, sigmoid vs. ReLU

Build a $25$-layer network, width $20$, with **identical weights** for both activation choices
(only $g$ differs) so the comparison isolates the activation function's own effect, matching
the lecture's argument.
Weights are drawn with a Xavier-like scale $\sqrt{1/K}$ so weight magnitude alone is not already
causing vanishing/exploding -- any shrinkage seen below comes from the activation's derivative,
not the weights.

Rather than run a full loss and backprop, seed the backward pass at the output layer with a fixed
vector and apply the lecture's recursion, $\delta^{(l)}=(W^{(l+1)}\delta^{(l+1)})\odot g'(\mathbf{z}^{(l)})$,
layer by layer, and track $\|\delta^{(l)}\|$ going backward toward the input (layer 1).

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_prime(z):
    return (z > 0).astype(float)

def build_deep_net(depth, width, seed, scale):
    rng_local = np.random.default_rng(seed)
    Ws = [rng_local.normal(scale=scale, size=(width, width)) for _ in range(depth)]
    bs = [np.zeros(width) for _ in range(depth)]
    return Ws, bs

def forward_deep(x0, Ws, bs, act):
    Zs, Xs = [], [x0]
    x = x0
    for W, b in zip(Ws, bs):
        z = x @ W + b
        x = act(z)
        Zs.append(z)
        Xs.append(x)
    return Zs, Xs

def backward_deep(Zs, Ws, delta_out, act_prime):
    """the lecture's recursion, applied layer by layer, seeded at the output."""
    deltas = [None] * len(Ws)
    deltas[-1] = delta_out
    for l in range(len(Ws) - 2, -1, -1):
        deltas[l] = (deltas[l + 1] @ Ws[l + 1].T) * act_prime(Zs[l])
    return deltas

depth, width = 25, 20
x0 = np.random.default_rng(1).normal(size=(1, width))
scale = np.sqrt(1.0 / width)  # Xavier-like: isolates the activation's own contribution

Ws, bs = build_deep_net(depth, width, seed=42, scale=scale)  # SAME weights for both activations

Zs_sig, _ = forward_deep(x0, Ws, bs, sigmoid)
Zs_relu, _ = forward_deep(x0, Ws, bs, relu)

delta_out = np.ones((1, width))  # same fixed seed vector for both

deltas_sig = backward_deep(Zs_sig, Ws, delta_out, sigmoid_prime)
deltas_relu = backward_deep(Zs_relu, Ws, delta_out, relu_prime)

norms_sig = np.array([np.linalg.norm(d) for d in deltas_sig])
norms_relu = np.array([np.linalg.norm(d) for d in deltas_relu])

print(f"layer 1  (earliest, closest to input): sigmoid ||delta|| = {norms_sig[0]:.3e},  relu ||delta|| = {norms_relu[0]:.3e}")
print(f"layer {depth} (last, closest to output):   sigmoid ||delta|| = {norms_sig[-1]:.3e},  relu ||delta|| = {norms_relu[-1]:.3e}")
print(f"\nsigmoid attenuation, layer {depth} -> layer 1: {norms_sig[-1] / norms_sig[0]:.3e}x")
print(f"relu attenuation,    layer {depth} -> layer 1: {norms_relu[-1] / norms_relu[0]:.3e}x")
print(f"\nratio of ReLU's to sigmoid's layer-1 gradient: {norms_relu[0] / norms_sig[0]:.3e}")
print("(ReLU also attenuates somewhat -- roughly half its units are inactive at any layer --")
print(" but nowhere near as catastrophically as sigmoid.)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
layers = np.arange(1, depth + 1)
ax.semilogy(layers, norms_sig, marker='o', label='sigmoid (saturating)')
ax.semilogy(layers, norms_relu, marker='s', label='ReLU (non-saturating)')
ax.set_xlabel('layer (1 = earliest, closest to input)')
ax.set_ylabel(r'$\|\delta^{(l)}\|$ (log scale)')
ax.set_title('Vanishing gradients: error-signal norm by layer')
ax.legend()
ax.invert_xaxis()
fig.tight_layout()
plt.show()

**Result:** with identical weights, the sigmoid network's earliest-layer gradient is attenuated
by a factor of roughly $10^{15}$ relative to the output layer, while ReLU's is attenuated by only
about $10^4$ over the same 25 layers -- ReLU's gradient signal reaching the earliest layer is
about $10^{11}$ times larger than sigmoid's.
This is the lecture's vanishing-gradient claim, verified directly rather than only asserted.

## Part 3: GD vs. momentum vs. Adam

The **Rosenbrock function**, $f(x,y) = (a-x)^2 + b(y-x^2)^2$ with the standard $a=1$, $b=100$,
is a classic non-convex test surface: a narrow, curved valley with very different curvature along
its length than across it -- exactly the lecture's condition-number story made concrete.
Its unique global minimum is at $(a,a^2)=(1,1)$, where $f=0$.
The objective is deterministic, so the baseline here is plain (full-batch) gradient
descent -- with minibatch gradients the same update would be SGD.
All three optimizers start at the same point, $w_0=(-1.5, 2.5)$.

In [ ]:
a, b = 1.0, 100.0

def rosenbrock(w):
    x, y = w
    return (a - x) ** 2 + b * (y - x ** 2) ** 2

def rosenbrock_grad(w):
    x, y = w
    dx = -2 * (a - x) - 4 * b * x * (y - x ** 2)
    dy = 2 * b * (y - x ** 2)
    return np.array([dx, dy])

w_start = np.array([-1.5, 2.5])
n_iters = 5000

def run_gd(eta, n_iters):
    w = w_start.copy()
    traj, losses = [w.copy()], [rosenbrock(w)]
    for _ in range(n_iters):
        w = w - eta * rosenbrock_grad(w)
        losses.append(rosenbrock(w))
        traj.append(w.copy())
    return w, losses, np.array(traj)

def run_momentum(eta, rho, n_iters):
    """the lecture's heavy-ball update."""
    w = w_start.copy()
    v = np.zeros_like(w)
    traj, losses = [w.copy()], [rosenbrock(w)]
    for _ in range(n_iters):
        g = rosenbrock_grad(w)
        v = rho * v + g
        w = w - eta * v
        losses.append(rosenbrock(w))
        traj.append(w.copy())
    return w, losses, np.array(traj)

def run_adam(eta, n_iters, beta1=0.9, beta2=0.999, eps=1e-8):
    """the lecture's Adam update, with the paper's default beta1, beta2, eps."""
    w = w_start.copy()
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    traj, losses = [w.copy()], [rosenbrock(w)]
    for t in range(1, n_iters + 1):
        g = rosenbrock_grad(w)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)
        mhat = m / (1 - beta1 ** t)
        vhat = v / (1 - beta2 ** t)
        w = w - eta * mhat / (np.sqrt(vhat) + eps)
        losses.append(rosenbrock(w))
        traj.append(w.copy())
    return w, losses, np.array(traj)

eta_gd = 0.001
eta_mom, rho = 0.001, 0.9
eta_adam = 0.01

w_gd, losses_gd, traj_gd = run_gd(eta_gd, n_iters)
w_mom, losses_mom, traj_mom = run_momentum(eta_mom, rho, n_iters)
w_adam, losses_adam, traj_adam = run_adam(eta_adam, n_iters)

print(f"After {n_iters} iterations from w0={w_start}, true optimum w*=({a},{a**2}):\n")
print(f"  GD       (eta={eta_gd}):            final loss = {losses_gd[-1]:.4e},  w = {w_gd}")
print(f"  Momentum (eta={eta_mom}, rho={rho}):  final loss = {losses_mom[-1]:.4e},  w = {w_mom}")
print(f"  Adam     (eta={eta_adam}):              final loss = {losses_adam[-1]:.4e},  w = {w_adam}")

### 3.1 Plain GD is not broken, just slow

Give plain GD $10\times$ more iterations at the same learning rate: it does eventually reach
essentially the same loss momentum and Adam reached in $1/10$ the steps -- confirming this is a
speed difference (the lecture's ill-conditioning), not a correctness difference.

In [ ]:
w_gd_long, losses_gd_long, _ = run_gd(eta_gd, 50000)
print(f"GD with 50000 iterations (10x more): final loss = {losses_gd_long[-1]:.4e}, w = {w_gd_long}")

### 3.2 Curvature caps the learning rate directly

Push GD's learning rate up to $\eta=0.005$ (still a small-looking number) and it diverges within
a handful of steps -- the lecture's claim that curvature puts a hard ceiling on a stable
learning rate, made concrete.

In [ ]:
with np.errstate(all='ignore'):
    w_div, losses_div, _ = run_gd(0.005, 10)
print(f"GD, eta=0.005: loss after 10 steps = {losses_div[-1]:.3e} (diverges, vs. eta=0.001's stable descent above)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.semilogy(losses_gd, label=f'GD (eta={eta_gd})')
ax.semilogy(losses_mom, label=f'Momentum (eta={eta_mom}, rho={rho})')
ax.semilogy(losses_adam, label=f'Adam (eta={eta_adam})')
ax.set_xlabel('iteration')
ax.set_ylabel('Rosenbrock loss (log scale)')
ax.set_title('GD vs. momentum vs. Adam: loss curves')
ax.legend()

ax = axes[1]
xs = np.linspace(-2, 2, 300)
ys = np.linspace(-1, 3, 300)
Xg, Yg = np.meshgrid(xs, ys)
Zg = (a - Xg) ** 2 + b * (Yg - Xg ** 2) ** 2
ax.contour(Xg, Yg, np.log10(Zg + 1e-8), levels=30, cmap='Greys', linewidths=0.5)
ax.plot(traj_gd[:, 0], traj_gd[:, 1], label='GD', linewidth=1.2)
ax.plot(traj_mom[:, 0], traj_mom[:, 1], label='Momentum', linewidth=1.2)
ax.plot(traj_adam[:, 0], traj_adam[:, 1], label='Adam', linewidth=1.2)
ax.plot(a, a ** 2, 'k*', markersize=12, label='optimum')
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Trajectories on the Rosenbrock valley')
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

**Result:** after the same $5{,}000$ iterations, momentum and Adam both reach loss below
$10^{-17}$ (essentially the exact optimum $(1,1)$), while plain GD at the largest stable learning
rate tested is still at loss $\approx 7\times10^{-3}$ -- visibly still descending the valley in the
trajectory plot, not yet at the optimum.
Momentum and Adam do not just converge to a *different* place; on this ill-conditioned surface
they converge to the *same* place dramatically faster.

## Summary

All three claims from Lecture 07 verified directly, not just asserted:

1. **Backpropagation computes the exact gradient.** A from-scratch implementation matches
   numerical differentiation to about $10^{-10}$ relative error.
2. **Vanishing gradients are real and activation-dependent.** With identical weights, a
   25-layer sigmoid network's earliest-layer gradient is about $10^{11}$ times smaller than the
   same network built with ReLU.
3. **Momentum and Adam converge faster than plain GD on an ill-conditioned, non-convex
   surface.** Both reach the Rosenbrock optimum in $5{,}000$ iterations; GD needs about
   $10\times$ more at the same learning rate, and a learning rate only $5\times$ larger makes it
   diverge outright -- the lecture's step-size ceiling, directly visible.